In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pydeseq2 as pydeseq2

In [2]:
orig_rna_soybean = sc.read_h5ad(
    r"C:\Users\mikep\git\Soybean_Single_Cell_drought_heat\Data\anndata_export\adata_rna.h5ad"
)

In [3]:
# .copy() so that normalising .X below does not also overwrite layers["counts"]
# (a bare assignment binds .X and the layer to the same matrix object)
orig_rna_soybean.X = orig_rna_soybean.layers["counts"].copy()
orig_rna_soybean

AnnData object with n_obs × n_vars = 30467 × 36042
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'pctCP', 'pctMT', 'nUMI_raw', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.3', 'SCT_snn_res.0.4', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6', 'SCT_snn_res.0.7', 'SCT_snn_res.0.8', 'SCT_snn_res.0.9', 'SCT_snn_res.1', 'seurat_clusters', 'treatment', 'libraries', 'replicate', 'source_rds', 'integrated_snn_res.0.5', 'celltype_call', 'cluster_annot'
    var: 'highly_variable', 'in_integrated'
    uns: 'neighbors', 'pca', 'pca_loadings', 'source'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'
    layers: 'counts', None (.X)

In [4]:
sc.pp.normalize_total(orig_rna_soybean, target_sum=1e4)
sc.pp.log1p(orig_rna_soybean)

In [5]:
orig_rna_soybean

AnnData object with n_obs × n_vars = 30467 × 36042
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'pctCP', 'pctMT', 'nUMI_raw', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.3', 'SCT_snn_res.0.4', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6', 'SCT_snn_res.0.7', 'SCT_snn_res.0.8', 'SCT_snn_res.0.9', 'SCT_snn_res.1', 'seurat_clusters', 'treatment', 'libraries', 'replicate', 'source_rds', 'integrated_snn_res.0.5', 'celltype_call', 'cluster_annot'
    var: 'highly_variable', 'in_integrated'
    uns: 'neighbors', 'pca', 'pca_loadings', 'source', 'log1p'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'
    layers: 'counts', None (.X)

In [6]:
del orig_rna_soybean.uns["neighbors"]
sc.pp.neighbors(orig_rna_soybean)

In [7]:
sc.tl.leiden(
    orig_rna_soybean,
    resolution=40,
    key_added="Ultra_high",
    flavor="igraph",
)

In [8]:
print(orig_rna_soybean.obs["Ultra_high"].value_counts().head(10))
print(print(orig_rna_soybean.obs["Ultra_high"].value_counts().tail(10)))

Ultra_high
141    144
10     142
389    140
48     138
461    138
345    134
378    134
448    134
218    133
329    131
Name: count, dtype: int64
Ultra_high
466    13
303    12
24     11
244    11
234    10
355    10
392     8
406     7
497     6
344     5
Name: count, dtype: int64
None


---

# Pseudobulk differential expression with PyDESeq2

Three complementary analyses, all built from **raw integer counts**:

1. **Library-level pseudobulk** — one pseudosample per library (4 Control, 3 Drought, 3 Heat,
   3 HD), design `~ treatment`. This is the statistically conservative reference analysis:
   its replicates are true biological replicates.
2. **Leiden micro-cluster pseudobulk** — leiden is re-run separately within each library and
   cells are pooled by micro-cluster, giving many small pseudo-replicates per treatment,
   design `~ treatment`.
3. **Multifactor** — the same micro-clusters, but also split by `celltype_call` and modelled
   with `~ celltype_call + treatment`, so the treatment effect is estimated *after* adjusting
   for cell-type composition.

Each analysis contrasts Control against Drought, Heat and HD.

> **Counts.** DESeq2 needs raw integer counts. `load_raw_counts()` below uses
> `layers["counts"]` when it still holds integers, and otherwise re-reads the matrix from the
> h5ad (needed if `.X` was normalised in place before the layer was copied).

> **Caveat for analyses 2 and 3.** Micro-clusters drawn from the same library are not
> independent biological replicates, so their p-values are anti-conservative compared with
> analysis 1. Treat analysis 1 as the reference and 2/3 as higher-resolution views; the
> concordance cell below quantifies the difference. Library cannot be added to the design as a
> covariate to absorb the batch effect, because library is nested within treatment.

In [9]:
from pathlib import Path

import anndata as ad
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from scipy import sparse

RAW_H5AD = Path(
    r"C:\Users\mikep\git\Soybean_Single_Cell_drought_heat\Data\anndata_export\adata_rna.h5ad"
)
RESULTS_DIR = Path("de_results")

TREATMENTS = [
    "Control",
    "Drought",
    "Heat",
    "HD",
]  # Control first -> DESeq2 reference level
CONDITIONS = ["Drought", "Heat", "HD"]  # each contrasted against Control

LEIDEN_KEY = "leiden_within_library"  # micro-clusters, computed one library at a time
LEIDEN_RES = 9  # resolution for that clustering (see the note above)

MIN_GENE_TOTAL = 10  # gene filter: total counts across pseudosamples
MIN_GENE_FRAC = 0.10  # gene filter: fraction of pseudosamples with a non-zero count
ALPHA, LFC_CUT = 0.05, 1.0  # thresholds used for the significance summaries
N_CPUS = 8
SAVE_RESULTS = True

CONTRASTS = {f"{c} vs Control": ["treatment", c, "Control"] for c in CONDITIONS}

In [10]:
def load_raw_counts(adata, path=RAW_H5AD, layer="counts"):
    """AnnData of raw integer counts, carrying `adata`'s obs (treatment, leiden, cell type)."""

    def is_integer(mat):
        data = mat.data if sparse.issparse(mat) else np.asarray(mat).ravel()
        return bool(np.allclose(data, np.rint(data)))

    mat = adata.layers[layer] if layer in adata.layers else None
    if mat is None or not is_integer(mat):
        print(
            f"layers['{layer}'] does not hold integers -- re-reading counts from {path.name}"
        )
        mat = sc.read_h5ad(path)[adata.obs_names, adata.var_names].layers[layer]
        if not is_integer(mat):
            raise ValueError(f"no raw integer counts in layers['{layer}']")
    counts = ad.AnnData(X=mat.copy(), obs=adata.obs.copy(), var=adata.var.copy())
    counts.obs["treatment"] = pd.Categorical(
        counts.obs["treatment"].astype(str), categories=TREATMENTS
    )
    return counts


def pseudobulk(adata, group_keys, meta_keys=()):
    """Sum raw counts within each group of cells.

    Returns (counts, metadata): a pseudosamples x genes integer count matrix, and a matching
    metadata frame holding the grouping columns, the majority value of each `meta_keys` column,
    and the number of cells per pseudosample.
    """
    group_keys = list(group_keys)
    labels = adata.obs[group_keys].astype(str).agg(" | ".join, axis=1).to_numpy()
    cats = pd.Categorical(labels)
    n_cells = np.bincount(cats.codes, minlength=len(cats.categories))

    indicator = sparse.csr_matrix(
        (np.ones(adata.n_obs), (cats.codes, np.arange(adata.n_obs))),
        shape=(len(cats.categories), adata.n_obs),
    )
    summed = indicator @ adata.X
    summed = (
        np.asarray(summed.todense()) if sparse.issparse(summed) else np.asarray(summed)
    )
    counts = pd.DataFrame(
        np.rint(summed).astype(np.int64),
        index=list(cats.categories),
        columns=adata.var_names,
    )

    meta_cols = group_keys + [k for k in meta_keys if k not in group_keys]
    frame = adata.obs[meta_cols].astype(str).copy()
    frame["_group"] = labels
    meta = frame.groupby("_group", observed=True)[meta_cols].agg(
        lambda s: s.value_counts().index[0]
    )
    meta["n_cells"] = pd.Series(n_cells, index=list(cats.categories))
    if "treatment" in meta:
        meta["treatment"] = pd.Categorical(meta["treatment"], categories=TREATMENTS)
    return counts, meta.loc[counts.index]


def filter_genes(counts, min_total=MIN_GENE_TOTAL, min_frac_samples=MIN_GENE_FRAC):
    """Drop genes with too little signal to be testable (also keeps PyDESeq2 fast)."""
    min_samples = max(2, int(np.ceil(min_frac_samples * counts.shape[0])))
    keep = ((counts > 0).sum(axis=0) >= min_samples) & (counts.sum(axis=0) >= min_total)
    print(f"genes kept: {int(keep.sum())} / {counts.shape[1]}")
    return counts.loc[:, keep.to_numpy()]

In [11]:
def run_deseq(counts, meta, design, contrasts=None, n_cpus=N_CPUS, **kwargs):
    """Fit one DESeq2 model and run a Wald test for each contrast."""
    contrasts = CONTRASTS if contrasts is None else contrasts
    dds = DeseqDataSet(
        counts=counts, metadata=meta, design=design, n_cpus=n_cpus, quiet=True, **kwargs
    )
    dds.deseq2()
    results = {}
    for name, contrast in contrasts.items():
        stat = DeseqStats(dds, contrast=contrast, n_cpus=n_cpus, quiet=True)
        stat.summary()
        results[name] = stat.results_df.sort_values("padj")
    return dds, results


def sig_genes(res, alpha=ALPHA, lfc_cut=LFC_CUT):
    return res.index[(res["padj"] < alpha) & (res["log2FoldChange"].abs() >= lfc_cut)]


def de_summary(results, alpha=ALPHA, lfc_cut=LFC_CUT):
    rows = []
    for name, res in results.items():
        sig = res.loc[sig_genes(res, alpha, lfc_cut)]
        rows.append(
            {
                "comparison": name,
                "genes_tested": int(res["padj"].notna().sum()),
                f"padj<{alpha}": int((res["padj"] < alpha).sum()),
                f"padj<{alpha} & |LFC|>={lfc_cut}": len(sig),
                "up": int((sig["log2FoldChange"] > 0).sum()),
                "down": int((sig["log2FoldChange"] < 0).sum()),
            }
        )
    return pd.DataFrame(rows).set_index("comparison")


def top_genes(results, n=10):
    """Top `n` genes by adjusted p-value for each comparison, side by side."""
    frames = []
    for name, res in results.items():
        top = res.dropna(subset=["padj"]).head(n)
        frames.append(
            pd.DataFrame(
                {
                    (name, "gene"): top.index,
                    (name, "log2FC"): top["log2FoldChange"].round(2).to_numpy(),
                    (name, "padj"): top["padj"].to_numpy(),
                }
            ).reset_index(drop=True)
        )
    out = pd.concat(frames, axis=1)
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    return out


def save_results(results, prefix):
    if not SAVE_RESULTS:
        return
    RESULTS_DIR.mkdir(exist_ok=True)
    for name, res in results.items():
        res.to_csv(RESULTS_DIR / f"{prefix}__{name.replace(' ', '_')}.csv")
    print(f"saved {len(results)} tables to {RESULTS_DIR.resolve()}")

In [12]:
counts_adata = load_raw_counts(orig_rna_soybean)
print(counts_adata)
print("\ncells per treatment / library:")
print(counts_adata.obs.groupby(["treatment", "libraries"], observed=True).size())

AnnData object with n_obs × n_vars = 30467 × 36042
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'pctCP', 'pctMT', 'nUMI_raw', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.3', 'SCT_snn_res.0.4', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6', 'SCT_snn_res.0.7', 'SCT_snn_res.0.8', 'SCT_snn_res.0.9', 'SCT_snn_res.1', 'seurat_clusters', 'treatment', 'libraries', 'replicate', 'source_rds', 'integrated_snn_res.0.5', 'celltype_call', 'cluster_annot', 'Ultra_high'
    var: 'highly_variable', 'in_integrated'
    layers: None (.X)

cells per treatment / library:
treatment  libraries         
Control    leaf_control_rep1     3740
           leaf_control_rep1A    3452
           leaf_control_rep2      823
           leaf_control_rep3     1683
Drought    leaf_drought_rep1     1434
           leaf_drought_rep2     2855
           leaf_drought_rep3     1036
Heat       leaf_Heat_rep1         939
           leaf_Heat_rep2        2381
           leaf_Heat_rep3        2193
HD         leaf_HD_rep1

## 1. Library-level pseudobulk (`~ treatment`)

One pseudosample per library: 4 Control, 3 Drought, 3 Heat, 3 HD. With only a handful of
replicates, genes are required to be detected in at least half of the libraries.

In [13]:
lib_counts, lib_meta = pseudobulk(
    counts_adata, group_keys=["libraries"], meta_keys=["treatment", "replicate"]
)
print(lib_counts.shape)
lib_meta[["treatment", "replicate", "n_cells"]]

(13, 36042)


,treatment,replicate,n_cells
leaf_HD_rep1,HD,rep1,5538
leaf_HD_rep2,HD,rep2,3691
leaf_HD_rep3,HD,rep3,702
leaf_Heat_rep1,Heat,rep1,939
leaf_Heat_rep2,Heat,rep2,2381
leaf_Heat_rep3,Heat,rep3,2193
leaf_control_rep1,Control,rep1,3740
leaf_control_rep1A,Control,rep1A,3452
leaf_control_rep2,Control,rep2,823
leaf_control_rep3,Control,rep3,1683


In [14]:
lib_counts_f = filter_genes(lib_counts, min_frac_samples=0.5)
lib_dds, lib_results = run_deseq(lib_counts_f, lib_meta, design="~treatment")
de_summary(lib_results)

genes kept: 28589 / 36042


,genes_tested,padj<0.05,padj<0.05 & |LFC|>=1.0,up,down
comparison,,,,,
Drought vs Control,19697,12,12,2,10
Heat vs Control,9731,8,8,7,1
HD vs Control,27453,2589,2398,1083,1315


In [15]:
save_results(lib_results, "library_pseudobulk")
top_genes(lib_results)

saved 3 tables to C:\Users\mikep\git\Soybean_Single_Cell_drought_heat\Initial_exploration\de_results


Drought vs Control                              Heat vs Control  \
                         gene log2FC      padj                        gene   
0  Glyma.10G058200.Wm82.a4.v1  -2.51  0.000019  Glyma.07G151800.Wm82.a4.v1   
1  Glyma.03G161300.Wm82.a4.v1  -3.79  0.001064  Glyma.01G238600.Wm82.a4.v1   
2  Glyma.14G066000.Wm82.a4.v1  -1.64  0.003243  Glyma.14G096800.Wm82.a4.v1   
3  Glyma.11G092900.Wm82.a4.v1  -1.64  0.029984  Glyma.07G061500.Wm82.a4.v1   
4  Glyma.15G055400.Wm82.a4.v1  -4.10  0.037394  Glyma.18G278900.Wm82.a4.v1   
5  Glyma.20G196900.Wm82.a4.v1  -2.32  0.039334  Glyma.17G252900.Wm82.a4.v1   
6  Glyma.15G070900.Wm82.a4.v1  -3.65  0.039334  Glyma.11G092900.Wm82.a4.v1   
7  Glyma.05G003500.Wm82.a4.v1   1.20  0.039334  Glyma.18G180800.Wm82.a4.v1   
8  Glyma.03G145500.Wm82.a4.v1  -2.09  0.039334  Glyma.15G077700.Wm82.a4.v1   
9  Glyma.04G056400.Wm82.a4.v1   1.57  0.039334  Glyma.06G170432.Wm82.a4.v1   

                                 HD vs Control                       
  log2FC      padj                        gene log2FC          padj  
0   5.13  0.001543  Glyma.01G238600.Wm82.a4.v1   7.55  3.976633e-11  
1   4.64  0.004561  Glyma.07G151800.Wm82.a4.v1   7.77  3.976633e-11  
2   4.76  0.004561  Glyma.14G010900.Wm82.a4.v1  -3.03  5.126979e-10  
3  10.10  0.015288  Glyma.19G222500.Wm82.a4.v1  -3.91  1.867350e-09  
4   3.08  0.018521  Glyma.18G024000.Wm82.a4.v1  -3.41  2.793596e-09  
5   2.53  0.040031  Glyma.11G092900.Wm82.a4.v1  -2.66  2.984852e-09  
6  -1.52  0.040031  Glyma.15G011200.Wm82.a4.v1  -5.13  1.279344e-08  
7   2.46  0.048451  Glyma.18G259700.Wm82.a4.v1   7.64  1.364125e-08  
8   3.82  0.069754  Glyma.07G228400.Wm82.a4.v1   5.41  1.793346e-08  
9   1.85  0.069977  Glyma.15G077700.Wm82.a4.v1   6.38  4.744669e-08

## 2. Leiden micro-cluster pseudobulk (`~ treatment`)

Leiden is re-run **one library at a time**. A clustering computed over all cells at once lets a
micro-cluster straddle libraries and conditions, and splitting such a cluster by treatment
afterwards only leaves unbalanced fragments of it (grouping the global resolution-40 clusters by
treatment gives 2386 groups, median 9 cells, smallest 1). Because libraries are separate
batches, clustering inside each one makes every group pure for library *and* treatment by
construction.

Resolution is relative to the size of the graph being clustered, so the value tuned on all
30,467 cells is far too high for a single 700-5,500 cell library — at resolution 50 the
libraries break into 4,327 groups with a median of 7 cells. `LEIDEN_RES = 5` gives ~615
pseudosamples (roughly 150 per treatment), median group sizes of 18-83 cells and a maximum of
194, i.e. the 10-400 range. Resolution 2 (~327 groups, medians 33-143) is the more conservative
alternative; the printout below reports what you actually got.

In [16]:
# re-cluster inside each library, so no micro-cluster can mix batches or conditions
micro_labels = pd.Series(index=counts_adata.obs_names, dtype=object)

for library in counts_adata.obs["libraries"].unique():
    sub = orig_rna_soybean[orig_rna_soybean.obs["libraries"] == library].copy()
    sub.uns.pop("neighbors", None)  # the stored graph belongs to the full object
    sc.pp.neighbors(sub, use_rep="X_pca")
    sc.tl.leiden(
        sub,
        resolution=LEIDEN_RES,
        key_added="_leiden",
        flavor="igraph",
        n_iterations=2,
    )
    micro_labels[sub.obs_names] = library + "_" + sub.obs["_leiden"].astype(str)

    sizes = sub.obs["_leiden"].value_counts()
    print(
        f"{library:>20}: {sub.n_obs:>5} cells -> {sizes.size:>3} clusters, "
        f"median {sizes.median():.0f}, range {sizes.min()}-{sizes.max()}"
    )

counts_adata.obs[LEIDEN_KEY] = pd.Categorical(micro_labels)
orig_rna_soybean.obs[LEIDEN_KEY] = pd.Categorical(
    micro_labels.reindex(orig_rna_soybean.obs_names)
)
print(
    f"\n{micro_labels.nunique()} micro-clusters, {micro_labels.isna().sum()} cells unassigned"
)

c:\Users\mikep\miniconda3\envs\Single_cell\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


   leaf_control_rep1:  3740 cells ->  90 clusters, median 42, range 6-100
  leaf_control_rep1A:  3452 cells ->  89 clusters, median 37, range 9-77
   leaf_control_rep2:   823 cells ->  66 clusters, median 11, range 2-26
   leaf_control_rep3:  1683 cells ->  78 clusters, median 19, range 5-51
   leaf_drought_rep1:  1434 cells ->  63 clusters, median 22, range 2-52
   leaf_drought_rep2:  2855 cells ->  78 clusters, median 35, range 2-84
   leaf_drought_rep3:  1036 cells ->  70 clusters, median 14, range 2-34
        leaf_HD_rep1:  5538 cells ->  93 clusters, median 58, range 12-117
        leaf_HD_rep2:  3691 cells ->  92 clusters, median 38, range 8-91
        leaf_HD_rep3:   702 cells ->  59 clusters, median 11, range 2-25
      leaf_Heat_rep1:   939 cells ->  57 clusters, median 17, range 4-32
      leaf_Heat_rep2:  2381 cells ->  81 clusters, median 29, range 7-63
      leaf_Heat_rep3:  2193 cells ->  75 clusters, median 29, range 6-68

991 micro-clusters, 0 cells unassigned


In [17]:
micro_counts, micro_meta = pseudobulk(
    counts_adata,
    group_keys=["treatment", LEIDEN_KEY],
    meta_keys=["celltype_call", "libraries"],
)
print(f"{micro_counts.shape[0]} pseudosamples x {micro_counts.shape[1]} genes")
print("\npseudosamples per treatment:")
print(micro_meta["treatment"].value_counts().reindex(TREATMENTS))
print("\ncells per pseudosample:")
print(micro_meta["n_cells"].describe().round(1))
micro_meta.head()

991 pseudosamples x 36042 genes

pseudosamples per treatment:
treatment
Control    323
Drought    211
Heat       213
HD         244
Name: count, dtype: int64

cells per pseudosample:
count    991.0
mean      30.7
std       20.0
min        2.0
25%       16.0
50%       26.0
75%       42.0
max      117.0
Name: n_cells, dtype: float64


,treatment,leiden_within_library,celltype_call,libraries,n_cells
Control | leaf_control_rep1A_0,Control,leaf_control_rep1A_0,Bundle sheath,leaf_control_rep1A,54
Control | leaf_control_rep1A_1,Control,leaf_control_rep1A_1,Epidermal,leaf_control_rep1A,54
Control | leaf_control_rep1A_10,Control,leaf_control_rep1A_10,Bundle sheath,leaf_control_rep1A,62
Control | leaf_control_rep1A_11,Control,leaf_control_rep1A_11,Bundle sheath,leaf_control_rep1A,47
Control | leaf_control_rep1A_12,Control,leaf_control_rep1A_12,Bundle sheath,leaf_control_rep1A,43


In [18]:
# hundreds of pseudosamples x ~20k genes -- this fit is the slowest step (minutes)
micro_counts_f = filter_genes(micro_counts)
micro_dds, micro_results = run_deseq(micro_counts_f, micro_meta, design="~treatment")
de_summary(micro_results)

genes kept: 24102 / 36042


,genes_tested,padj<0.05,padj<0.05 & |LFC|>=1.0,up,down
comparison,,,,,
Drought vs Control,18962,6692,1122,792,330
Heat vs Control,18027,6131,1073,257,816
HD vs Control,22700,12363,5007,1383,3624


In [19]:
save_results(micro_results, "leiden_micro_pseudobulk")
top_genes(micro_results)

saved 3 tables to C:\Users\mikep\git\Soybean_Single_Cell_drought_heat\Initial_exploration\de_results


Drought vs Control                        \
                         gene log2FC           padj   
0  Glyma.14G162100.Wm82.a4.v1   3.30  4.090602e-103   
1  Glyma.02G250700.Wm82.a4.v1  -1.92   1.566511e-91   
2  Glyma.03G261800.Wm82.a4.v1  -1.26   5.743573e-87   
3  Glyma.14G066000.Wm82.a4.v1  -1.59   7.670157e-86   
4  Glyma.08G308200.Wm82.a4.v1   2.36   1.586186e-80   
5  Glyma.05G029200.Wm82.a4.v1   4.43   2.797806e-80   
6  Glyma.19G138000.Wm82.a4.v1   1.99   2.066860e-75   
7  Glyma.06G021200.Wm82.a4.v1   2.23   1.961528e-70   
8  Glyma.15G070900.Wm82.a4.v1  -3.41   1.122287e-68   
9  Glyma.14G130700.Wm82.a4.v1   0.81   2.421292e-62   

              Heat vs Control                        \
                         gene log2FC           padj   
0  Glyma.02G250700.Wm82.a4.v1  -2.20  1.409077e-121   
1  Glyma.11G232500.Wm82.a4.v1   1.69  7.041666e-102   
2  Glyma.17G202700.Wm82.a4.v1   1.58  1.862117e-101   
3  Glyma.19G219100.Wm82.a4.v1   3.71   9.530259e-83   
4  Glyma.14G130700.Wm82.a4.v1   0.90   6.405329e-80   
5  Glyma.06G117800.Wm82.a4.v1   1.72   1.357041e-73   
6  Glyma.11G223000.Wm82.a4.v1  -1.76   2.145824e-71   
7  Glyma.07G151800.Wm82.a4.v1   3.67   5.059950e-71   
8  Glyma.03G025600.Wm82.a4.v1   1.76   2.699194e-62   
9  Glyma.14G066000.Wm82.a4.v1  -1.28   8.648290e-62   

                HD vs Control                        
                         gene log2FC           padj  
0  Glyma.01G141900.Wm82.a4.v1   3.67   0.000000e+00  
1  Glyma.03G025600.Wm82.a4.v1   4.40   0.000000e+00  
2  Glyma.18G034600.Wm82.a4.v1  -3.74   0.000000e+00  
3  Glyma.16G101000.Wm82.a4.v1   4.36  8.638073e-280  
4  Glyma.07G151800.Wm82.a4.v1   6.77  2.702626e-271  
5  Glyma.02G250700.Wm82.a4.v1  -3.18  4.398247e-266  
6  Glyma.01G241400.Wm82.a4.v1   5.05  8.735374e-265  
7  Glyma.19G219100.Wm82.a4.v1   6.17  4.846374e-255  
8  Glyma.11G037100.Wm82.a4.v1  -3.72  9.828217e-247  
9  Glyma.07G222900.Wm82.a4.v1   2.62  4.196690e-242

In [20]:
# how far do the micro-cluster calls track the conservative library-level analysis?
rows = []
for name in CONTRASTS:
    lib_res, mic_res = lib_results[name], micro_results[name]
    lib_sig, mic_sig = set(sig_genes(lib_res)), set(sig_genes(mic_res))
    shared = lib_res.index.intersection(mic_res.index)
    rows.append(
        {
            "comparison": name,
            "library sig": len(lib_sig),
            "micro sig": len(mic_sig),
            "shared": len(lib_sig & mic_sig),
            "% of library sig recovered": round(
                100 * len(lib_sig & mic_sig) / max(len(lib_sig), 1), 1
            ),
            "LFC spearman (shared genes)": round(
                lib_res.loc[shared, "log2FoldChange"].corr(
                    mic_res.loc[shared, "log2FoldChange"], method="spearman"
                ),
                3,
            ),
        }
    )
pd.DataFrame(rows).set_index("comparison")

,library sig,micro sig,shared,% of library sig recovered,LFC spearman (shared genes)
comparison,,,,,
Drought vs Control,12,1122,12,100.0,0.929
Heat vs Control,8,1073,8,100.0,0.885
HD vs Control,2398,5007,1843,76.9,0.943


## 3. Multifactor: cell type as a second factor (`~ celltype_call + treatment`)

The micro-clusters are rebuilt so each one is also homogeneous for `celltype_call` (leiden
clusters at this resolution are already close to cell-type pure). Cell type then enters the
design as a covariate, so the treatment coefficients describe the response *within* cell types
instead of being confounded by shifts in cell-type composition between libraries.

Cell types without at least two pseudosamples in every treatment are dropped, so that all
factor levels stay estimable.

In [21]:
mf_counts, mf_meta = pseudobulk(
    counts_adata,
    group_keys=["treatment", "celltype_call", LEIDEN_KEY],
    meta_keys=["libraries"],
)

balance = pd.crosstab(mf_meta["celltype_call"], mf_meta["treatment"])
keep_ct = balance.index[(balance >= 2).all(axis=1)]
dropped = balance.index.difference(keep_ct)
if len(dropped):
    print(
        f"dropped cell types with <2 pseudosamples in some treatment: {list(dropped)}"
    )

mf_meta = mf_meta[mf_meta["celltype_call"].isin(keep_ct)].copy()
mf_meta["celltype_call"] = pd.Categorical(
    mf_meta["celltype_call"], categories=sorted(keep_ct)
)
mf_counts = mf_counts.loc[mf_meta.index]

print(f"\n{mf_counts.shape[0]} pseudosamples x {mf_counts.shape[1]} genes")
print("cells per pseudosample:", mf_meta["n_cells"].describe().round(1).to_dict())
balance.loc[keep_ct]


1485 pseudosamples x 36042 genes
cells per pseudosample: {'count': 1485.0, 'mean': 20.5, 'std': 20.5, 'min': 1.0, '25%': 3.0, '50%': 15.0, '75%': 31.0, 'max': 117.0}


treatment,Control,Drought,Heat,HD
celltype_call,,,,
Bundle sheath,82,49,58,41
Companion cells,35,25,22,20
Epidermal,49,35,34,30
Epidermal/Pavement,34,10,12,5
Guard cells,17,15,9,7
Hydathode cells,14,7,9,9
Mesophyll,245,161,162,193
Unknown,9,7,7,6
Vascular/Phloem parenchyma,17,14,20,16


In [22]:
mf_counts_f = filter_genes(mf_counts)
mf_dds, mf_results = run_deseq(
    mf_counts_f, mf_meta, design="~celltype_call + treatment"
)
de_summary(mf_results)

genes kept: 21470 / 36042


c:\Users\mikep\miniconda3\envs\Single_cell\Lib\site-packages\pydeseq2\dds.py:539: UserWarning: Every gene contains at least one zero, cannot compute log geometric means. Switching to iterative mode.
  self.fit_size_factors(


KeyboardInterrupt: 

In [ ]:
save_results(mf_results, "leiden_micro_multifactor_celltype")
top_genes(mf_results)

In [ ]:
# the three designs side by side
overview = pd.concat(
    {
        "1. library (~treatment)": de_summary(lib_results),
        "2. micro-cluster (~treatment)": de_summary(micro_results),
        "3. multifactor (~celltype + treatment)": de_summary(mf_results),
    },
    names=["analysis"],
)
overview

## 4. Library-level pseudobulk *within* each cell type (`~ treatment`, one model per cell type)

Analysis 1 repeated separately inside every cell type: cells are pooled by
`celltype_call x libraries`, so each pseudosample is still one library (a true biological
replicate) and the design stays `~ treatment` — only the cells contributing to it are
restricted to a single cell type. This answers "which genes respond to the stress *in this cell
type*", with the same conservative replicate structure as analysis 1, rather than analysis 3's
composition-adjusted average effect across cell types.

Guards, since a library can contribute very few cells of a rare cell type:

* pseudosamples built from fewer than `MIN_CELLS_PER_PSEUDOSAMPLE` cells are dropped;
* a treatment needs at least `MIN_LIBS_PER_TREATMENT` surviving libraries to be modelled — a
  cell type keeps only such treatments, and is skipped entirely if Control or every stress
  condition falls out;
* genes must be detected in at least half of the remaining libraries (as in analysis 1).

Power is low by construction (3-4 replicates per treatment, split further by cell type), so
expect fewer calls than analyses 2 and 3 — a gene missing here is usually untested, not
disproved.

In [ ]:
MIN_CELLS_PER_PSEUDOSAMPLE = 10  # a library must contribute this many cells of the cell type
MIN_LIBS_PER_TREATMENT = 2  # a treatment needs this many libraries left to be modelled

ct_lib_counts, ct_lib_meta = pseudobulk(
    counts_adata,
    group_keys=["celltype_call", "libraries"],
    meta_keys=["treatment", "replicate"],
)
print(f"{ct_lib_counts.shape[0]} cell type x library pseudosamples before filtering")

ct_lib_meta = ct_lib_meta[ct_lib_meta["n_cells"] >= MIN_CELLS_PER_PSEUDOSAMPLE]
ct_lib_counts = ct_lib_counts.loc[ct_lib_meta.index]
print(
    f"{ct_lib_counts.shape[0]} left with >= {MIN_CELLS_PER_PSEUDOSAMPLE} cells "
    f"(median {ct_lib_meta['n_cells'].median():.0f} cells, "
    f"max {ct_lib_meta['n_cells'].max()})\n"
)

print("libraries per cell type and treatment:")
libs_per_ct = pd.crosstab(ct_lib_meta["celltype_call"], ct_lib_meta["treatment"])
libs_per_ct

In [ ]:
# one independent analysis-1 fit per cell type (a dozen small models -- a minute or so)
ct_dds, ct_results = {}, {}

for celltype in libs_per_ct.index:
    meta = ct_lib_meta[ct_lib_meta["celltype_call"] == celltype].copy()
    n_libs = meta["treatment"].value_counts()
    usable = [t for t in TREATMENTS if n_libs.get(t, 0) >= MIN_LIBS_PER_TREATMENT]
    if "Control" not in usable or len(usable) < 2:
        print(
            f"--- {celltype}: skipped, only {dict(n_libs[n_libs > 0])} libraries pass the filter"
        )
        continue

    meta = meta[meta["treatment"].isin(usable)]
    # re-level so Control is still the first (reference) category after dropping treatments
    meta["treatment"] = pd.Categorical(meta["treatment"].astype(str), categories=usable)
    counts = ct_lib_counts.loc[meta.index]

    print(f"\n--- {celltype}: {dict(meta['treatment'].value_counts()[usable])}")
    counts = filter_genes(counts, min_frac_samples=0.5)
    dds, results = run_deseq(
        counts,
        meta,
        design="~treatment",
        contrasts={
            f"{c} vs Control": ["treatment", c, "Control"] for c in usable[1:]
        },
    )
    ct_dds[celltype], ct_results[celltype] = dds, results
    print(de_summary(results))

In [ ]:
# save one table per cell type x comparison, and stack the summaries
for celltype, results in ct_results.items():
    slug = celltype.replace(" ", "_").replace("/", "-")
    save_results(results, f"celltype_library_pseudobulk_{slug}")

ct_overview = pd.concat(
    {ct: de_summary(res) for ct, res in ct_results.items()}, names=["cell type"]
)
ct_overview

### Outputs

`de_results/` holds one CSV per analysis and comparison
(`library_pseudobulk__*`, `leiden_micro_pseudobulk__*`, `leiden_micro_multifactor_celltype__*`),
each a full PyDESeq2 results table (`baseMean`, `log2FoldChange`, `lfcSE`, `stat`, `pvalue`,
`padj`) sorted by adjusted p-value. A positive `log2FoldChange` means higher in the stressed
condition than in Control.

The fitted objects stay in memory as `lib_dds`, `micro_dds` and `mf_dds` for follow-up work —
e.g. LFC shrinkage, or per-cell-type contrasts by re-running analysis 3 one cell type at a time.